# Diabetes Diet Optimization
Maps Project 2 food prices to micronutrients from min_cost_data, then solves for the cheapest diet that meets diabetes nutritional requirements.

## Step 1 — Install & Import Libraries

In [1]:
# Install fuzzy matching library if not already installed
# %pip install thefuzz python-Levenshtein

import pandas as pd
import numpy as np
from thefuzz import process
from scipy.optimize import linprog
import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load Your Data

In [2]:
# Load Project 2 food prices (your 2023 retail prices)
project2 = pd.read_csv('Project_2_food_price - Sheet1(1).csv')
print('Project 2 shape:', project2.shape)
project2.head()

Project 2 shape: (176, 6)


,Food,Quantity,Units,Price,Year,FCD_id
0,white flour,1.0,pound,0.540,2023,NaN
1,rice long grain,1.0,pound,0.970,2023,NaN
2,spaghetti,1.0,pound,1.475,2023,NaN
3,white bread,1.0,pound,1.888,2023,NaN
4,whole wheat bread,1.0,pound,2.451,2023,NaN


In [3]:
# Load nutrients from min_cost_data
nutrients = pd.read_excel('min_cost_data(1).xlsx', sheet_name='nutrients')
print('Nutrients shape:', nutrients.shape)
nutrients.head()

Nutrients shape: (2750, 67)


,ingred_code,Ingredient description,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
0,1001,"Butter, salted",2.529,2.587,7.436,21.697,0.961,9.999,19.961,2.728,...,0.17,0.0,0.003,0.0,0.0,2.32,0.0,7.0,15.87,0.09
1,1002,"Butter, whipped, with salt",2.039,2.354,7.515,20.531,1.417,7.649,17.370,2.713,...,0.07,0.0,0.008,0.0,0.0,1.37,0.0,4.6,16.72,0.05
2,1003,"Butter oil, anhydrous",2.495,2.793,10.005,26.166,2.228,12.056,25.026,2.247,...,0.01,0.0,0.001,0.0,0.0,2.80,0.0,8.6,0.24,0.01
3,1004,"Cheese, blue",0.601,0.491,3.301,9.153,0.816,3.235,6.622,0.536,...,1.22,0.0,0.166,0.0,0.5,0.25,0.0,2.4,42.41,2.66
4,1005,"Cheese, brick",0.585,0.482,3.227,8.655,0.817,3.455,7.401,0.491,...,1.26,0.0,0.065,0.0,0.5,0.26,0.0,2.5,41.11,2.60


In [4]:
# Load RDA (dietary requirement constraints)
rda = pd.read_excel('min_cost_data(1).xlsx', sheet_name='rda')
print('RDA shape:', rda.shape)
rda

RDA shape: (27, 17)


,Nutrient,Nutrient Type,Unit,Constraint Type,Child_1_3,Female_4_8,Male_4_8,Female_9_13,Male_9_13,Female_14_18,Male_14_18,Female_19_30,Male_19_30,Female_31_50,Male_31_50,Female_51U,Male_51U
0,Energy,Macro,kcal,RDA,1000.00,1200.00,1400.00,1600.0,1800.0,1800.00,2200.00,2000.0,2400.0,1800.0,2200.0,1600.0,2000.0
1,Protein,Macro,g,RDA,13.00,19.00,19.00,34.0,34.0,46.00,52.00,46.0,56.0,46.0,56.0,46.0,56.0
2,Carbohydrate,Macro,g,RDA,130.00,130.00,130.00,130.0,130.0,130.00,130.00,130.0,130.0,130.0,130.0,130.0,130.0
3,Dietary Fiber,Macro,g,RDA,14.00,16.80,19.60,22.4,25.2,25.20,30.80,28.0,33.6,25.2,30.8,22.4,28.0
4,Linoleic Acid,Macro,g,AI,7.00,10.00,10.00,10.0,12.0,11.00,16.00,12.0,17.0,12.0,17.0,11.0,14.0
5,Linolenic Acid,Macro,g,AI,0.70,0.90,0.90,1.0,1.2,1.10,1.60,1.1,1.6,1.1,1.6,1.1,1.6
6,Calcium,Mineral,mg,RDA,700.00,1000.00,1000.00,1300.0,1300.0,1300.00,1300.00,1000.0,1000.0,1000.0,1000.0,1200.0,1200.0
7,Iron,Mineral,mg,RDA,7.00,10.00,10.00,8.0,8.0,15.00,11.00,18.0,8.0,18.0,8.0,8.0,8.0
8,Magnesium,Mineral,mg,RDA,80.00,130.00,130.00,240.0,240.0,360.00,410.00,310.0,400.0,320.0,420.0,320.0,420.0
9,Phosphorus,Mineral,mg,RDA,460.00,500.00,500.00,1250.0,1250.0,1250.00,1250.00,700.0,700.0,700.0,700.0,700.0,700.0


## Step 3 — Fuzzy Match Food Names
Project 2 uses names like `cheddar cheese`. Nutrients sheet uses `Cheese, cheddar`. Fuzzy matching bridges this gap automatically.

In [5]:
# Get food name lists from both sources
p2_foods    = project2['Food'].tolist()
nut_foods   = nutrients['Ingredient description'].tolist()

# Fuzzy match each Project 2 food to the closest nutrient description
matches = {}
for food in p2_foods:
    best_match, score = process.extractOne(food, nut_foods)
    matches[food] = {'matched_to': best_match, 'score': score}

match_df = pd.DataFrame(matches).T
match_df['score'] = match_df['score'].astype(int)
print(f'Total foods matched: {len(match_df)}')
match_df

Total foods matched: 175


,matched_to,score
white flour,"Egg, white, raw, fresh",86
rice long grain,"Babyfood, dinner, turkey and rice, strained",86
spaghetti,"Babyfood, dinner, spaghetti and tomato and mea...",90
white bread,"Fast foods, submarine sandwich, cold cut on wh...",90
whole wheat bread,"Bread, pita, whole-wheat",95
...,...,...
"Raspberries, fresh","Egg, duck, whole, fresh, raw",86
"Raspberries, frozen","Raspberries, frozen, red, sweetened",90
"Strawberries, fresh","Egg, goose, whole, fresh, raw",86
"Strawberries, frozen","Strawberries, frozen, unsweetened",90


In [6]:
# Review LOW confidence matches (score < 80) — these may need manual correction
low_confidence = match_df[match_df['score'] < 80]
print(f'Low confidence matches: {len(low_confidence)}')
low_confidence

Low confidence matches: 1


,matched_to,score
avacado,"Avocados, raw, all commercial varieties",77


In [7]:
# ---------------------------------------------------------------
# MANUAL CORRECTIONS — edit this dict based on low_confidence above
# Format: 'Project2 food name' : 'Correct nutrient description'
# ---------------------------------------------------------------
manual_fixes = {
    # Example corrections — update these after reviewing low_confidence above
    # 'avacado'          : 'Avocado, raw',
    # 'white flour'      : 'Wheat flour, white, all-purpose, unenriched',
    # 'grade A eggs'     : 'Egg, whole, raw, fresh',
}

# Apply manual fixes
for food, fix in manual_fixes.items():
    if food in match_df.index:
        match_df.loc[food, 'matched_to'] = fix
        match_df.loc[food, 'score'] = 100  # mark as manually confirmed

print('Manual fixes applied:', len(manual_fixes))

Manual fixes applied: 0


## Step 4 — Merge Prices with Nutrients

In [8]:
# Add matched nutrient name back to project2 dataframe
project2['matched_name'] = project2['Food'].map(
    match_df['matched_to'].to_dict()
)

# Merge project2 prices with nutrient data
merged = project2.merge(
    nutrients,
    left_on='matched_name',
    right_on='Ingredient description',
    how='inner'
)

print(f'Foods successfully matched with nutrients: {len(merged)}')
merged[['Food', 'Price', 'matched_name']].head(10)

Foods successfully matched with nutrients: 176


,Food,Price,matched_name
0,white flour,0.540,"Egg, white, raw, fresh"
1,rice long grain,0.970,"Babyfood, dinner, turkey and rice, strained"
2,spaghetti,1.475,"Babyfood, dinner, spaghetti and tomato and mea..."
3,white bread,1.888,"Fast foods, submarine sandwich, cold cut on wh..."
4,whole wheat bread,2.451,"Bread, pita, whole-wheat"
5,chocolate chip cookie,5.508,"Milk, chocolate, fluid, commercial, whole, wit..."
6,ground chuck beef,4.644,"Spices, mustard seed, ground"
7,ground beef,4.791,"Fast foods, nachos, with cheese, beans, ground..."
8,lean ground beef,6.393,"Spices, cinnamon, ground"
9,stew beef,6.773,"Babyfood, meat, beef, strained"


## Step 5 — Normalize Prices to Per-100g
Nutrients in the database are per 100g. Prices need to match the same unit.

In [9]:
# Convert all prices to price-per-100g
# Currently prices are per Quantity/Units (e.g. 1 pound, 16 oz, 1 gallon)

# Unit conversion to grams
unit_to_grams = {
    'pound'   : 453.592,
    'pound '  : 453.592,  # handle trailing space
    'oz'      : 28.3495,
    'gallon'  : 3785.41,  # for liquids (ml ≈ g for water-based)
    'egg'     : 50.0,     # approx per egg
}

def price_per_100g(row):
    unit  = str(row['Units']).strip().lower()
    qty   = row['Quantity']
    price = row['Price']
    grams_per_unit = unit_to_grams.get(unit, None)
    if grams_per_unit is None:
        return None  # unknown unit — will be dropped
    total_grams = qty * grams_per_unit
    return (price / total_grams) * 100  # price per 100g

merged['price_per_100g'] = merged.apply(price_per_100g, axis=1)

# Drop rows where we couldn't compute price
merged = merged.dropna(subset=['price_per_100g'])

# If same food appears multiple times, keep cheapest
merged = merged.sort_values('price_per_100g').drop_duplicates(subset='Food')

print(f'Foods with valid prices: {len(merged)}')
merged[['Food', 'Units', 'Price', 'price_per_100g']].head(10)

Foods with valid prices: 175


,Food,Units,Price,price_per_100g
151,"Papaya, fresh",pound,1.305513,0.017989
175,Watermelon,pound,0.405825,0.089469
25,low-fat milk,gallon,3.817000,0.100835
17,whole milk,gallon,4.204000,0.111058
0,white flour,pound,0.540000,0.119050
122,Bananas,pound,0.609595,0.134393
160,"Pineapple, fresh",pound,0.668427,0.147363
128,"Cantaloupe, fresh",pound,0.791924,0.174590
46,green cabbage,pound,0.826600,0.182234
21,white sugar,pound,0.860000,0.189598


## Step 6 — Build Matrix A and Price Vector p

In [10]:
# Nutrient columns (everything except food name/code columns)
non_nutrient_cols = ['Food', 'Quantity', 'Units', 'Price', 'Year', 'FCD_id',
                     'matched_name', 'ingred_code', 'Ingredient description',
                     'price_per_100g']

nutrient_cols = [c for c in merged.columns if c not in non_nutrient_cols]

# Set food name as index
merged = merged.set_index('Food')

# Matrix A: rows = foods, columns = nutrients (fill missing with 0)
A = merged[nutrient_cols].fillna(0)

# Price vector p
p = merged['price_per_100g']

print('Matrix A shape (foods x nutrients):', A.shape)
print('Price vector p length:', len(p))
A

Matrix A shape (foods x nutrients): (175, 65)
Price vector p length: 175


,Capric acid,Lauric acid,Myristic acid,Palmitic acid,Palmitoleic acid,Stearic acid,Oleic acid,Linoleic Acid,Linolenic Acid,Stearidonic acid,...,Vitamin B12,"Vitamin B-12, added",Vitamin B6,Vitamin C,Vitamin D,Vitamin E,"Vitamin E, added",Vitamin K,Water,Zinc
Food,,,,,,,,,,,,,,,,,,,,,
"Papaya, fresh",0.006,0.000,0.033,2.231,0.201,0.811,3.411,1.555,0.048,0.000,...,0.89,0.0,0.170,0.0,2.0,1.05,0.0,0.3,76.15,1.29
Watermelon,0.001,0.001,0.000,0.008,0.000,0.006,0.037,0.050,0.000,0.000,...,0.00,0.0,0.045,8.1,0.0,0.05,0.0,0.1,91.45,0.10
low-fat milk,0.004,0.002,0.017,0.378,0.006,0.399,0.455,0.050,0.002,0.000,...,0.14,0.0,0.023,0.9,0.1,0.20,0.0,0.6,61.43,0.43
whole milk,0.582,0.690,2.189,5.331,0.597,2.442,5.649,0.393,0.372,0.000,...,2.28,0.0,0.037,0.0,0.4,0.19,0.0,2.3,50.01,2.92
white flour,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.09,0.0,0.005,0.0,0.0,0.00,0.0,0.0,87.57,0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Apricots, dried",0.000,0.000,0.000,0.017,0.000,0.000,0.074,0.074,0.000,0.000,...,0.00,0.0,0.143,1.0,0.0,4.33,0.0,3.1,30.89,0.39
"Mangoes, dried",0.145,0.162,0.581,1.520,0.129,0.700,1.454,0.131,0.084,0.000,...,3.82,0.0,0.338,5.7,0.5,0.10,0.0,0.4,2.97,4.02
"steak, sirloin",0.008,0.008,0.178,1.682,0.211,0.736,2.257,0.152,0.059,0.000,...,1.18,0.0,0.596,0.0,0.1,0.31,0.0,1.2,68.72,4.12
